# Módulo 8 · Aprendizaje por Refuerzo (RL): de los MDP a RLHF

## Objetivos
1. Formalizar un problema como **Proceso de Decisión de Markov (MDP)** y resolverlo exactamente con **programación dinámica** (iteración de valores/políticas).
2. Aprender sin modelo con **diferencias temporales**: **Q-learning** y SARSA en entornos de **Gymnasium**.
3. Escalar a estados continuos con **Deep Q-Networks (DQN)**: *replay buffer*, red objetivo, ε-greedy.
4. Optimizar políticas directamente con **gradiente de política** (REINFORCE, línea base) y entender **actor-crítico** y **PPO**, el algoritmo de trabajo de la industria.
5. Conectar RL con los LLMs: **RLHF** (PPO sobre un modelo de recompensa), **DPO** y **GRPO** (razonamiento con recompensas verificables).
6. Conocer el ecosistema 2026: Gymnasium 1.x, Stable-Baselines3, CleanRL, TRL; simuladores (MuJoCo, Isaac Lab) y RL *offline*.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| MDPs finitos, retornos, funciones de valor, ecuaciones de Bellman | Sutton & Barto, *Reinforcement Learning: An Introduction* (2.ª ed., 2018) | **Cap. 3** Finite Markov Decision Processes |
| Programación dinámica (iteración de políticas y valores) | Sutton & Barto | **Cap. 4** Dynamic Programming |
| Métodos Monte Carlo | Sutton & Barto | **Cap. 5** Monte Carlo Methods |
| Aprendizaje TD: TD(0), SARSA, Q-learning | Sutton & Barto | **Cap. 6** Temporal-Difference Learning |
| Gradiente de política: REINFORCE, línea base, actor-crítico | Sutton & Barto | **Cap. 13** Policy Gradient Methods |
| Introducción práctica: Gym, políticas neuronales, DQN, PG | Géron, *Hands-On ML* (3.ª ed.) | **Cap. 18** Reinforcement Learning |
| RL en el marco de D2L | Zhang et al., *Dive into Deep Learning* | **Cap. 17** Reinforcement Learning (MDP, iteración de valores, Q-learning) |
| RLHF para LLMs | Fregly et al., *Generative AI on AWS* | **Cap. 7** Fine-Tuning with RLHF |

> **Nota de actualización (2026).** Se precisan los capítulos de Sutton & Barto (la malla decía "capítulos iniciales"). Géron Cap. 18 usa *OpenAI Gym*, hoy sustituido por **Gymnasium** (Farama Foundation; API `reset() -> (obs, info)` y `step() -> (obs, r, terminated, truncated, info)`). Se añade la sección RLHF/DPO/GRPO, que en 2026 es la aplicación de RL con más impacto económico.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 8 · Aprendizaje por Refuerzo**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [1]:
import numpy as np, matplotlib.pyplot as plt, time
from collections import deque
rng = np.random.default_rng(0)
try:
    import gymnasium as gym; GYM = True; print("Gymnasium", gym.__version__)
except ImportError:
    GYM = False; print("Gymnasium no instalado: `uv pip install gymnasium`")
try:
    import torch, torch.nn as nn, torch.nn.functional as F; TORCH = True; torch.manual_seed(0); print("PyTorch", torch.__version__)
except ImportError:
    TORCH = False; print("PyTorch no instalado")

Gymnasium 1.3.0


PyTorch 2.14.0+cu130


## 1. MDP y ecuaciones de Bellman (Sutton & Barto Cap. 3-4)

Un **MDP** es $(\mathcal S,\mathcal A, p, r, \gamma)$: en el estado $s$ el agente elige $a$, recibe recompensa $r$ y pasa a $s'$ con probabilidad $p(s'|s,a)$. El **retorno** es $G_t=\sum_k\gamma^k R_{t+k+1}$. Una **política** $\pi(a|s)$ induce funciones de valor

$$v_\pi(s) = \mathbb{E}_\pi[G_t\mid S_t=s],\qquad q_\pi(s,a)=\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a],$$

que satisfacen la **ecuación de Bellman** (recursión):
$$v_\pi(s)=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)\big[r+\gamma v_\pi(s')\big].$$
La **optimalidad de Bellman**: $v_*(s)=\max_a\sum_{s',r}p(s',r|s,a)[r+\gamma v_*(s')]$. Con el modelo $p$ conocido, **iteración de valores** aplica esta ecuación como operador hasta converger (es una contracción: convergencia garantizada). La política óptima es *greedy* respecto a $v_*$.

Construimos un *gridworld* con viento estocástico y lo resolvemos exactamente.

In [2]:
class GridWorld:
    '''Rejilla H x W. Acciones: 0=↑ 1=→ 2=↓ 3=←. Con prob. 'ruido' la acción se desvía a un lado. Meta +1, pozo -1, paso -0.02.'''
    def __init__(self, H=4, W=5, meta=(0, 4), pozo=(1, 4), muros=((1, 1), (2, 3)), ruido=0.2, gamma=0.95):
        self.H, self.W, self.meta, self.pozo, self.muros, self.ruido, self.gamma = H, W, meta, pozo, set(muros), ruido, gamma
        self.estados = [(i, j) for i in range(H) for j in range(W) if (i, j) not in self.muros]
        self.idx = {s: k for k, s in enumerate(self.estados)}; self.nS, self.nA = len(self.estados), 4
        self.terminal = {meta, pozo}
        self.P = self._construir_modelo()
    def _mover(self, s, a):
        if s in self.terminal: return s
        di, dj = [(-1, 0), (0, 1), (1, 0), (0, -1)][a]; ns = (s[0] + di, s[1] + dj)
        return s if not (0 <= ns[0] < self.H and 0 <= ns[1] < self.W) or ns in self.muros else ns
    def recompensa(self, s):
        return 1.0 if s == self.meta else -1.0 if s == self.pozo else -0.02
    def _construir_modelo(self):
        P = np.zeros((self.nS, self.nA, self.nS))
        for s in self.estados:
            for a in range(4):
                for a_real, pr in [(a, 1 - self.ruido), ((a - 1) % 4, self.ruido / 2), ((a + 1) % 4, self.ruido / 2)]:
                    P[self.idx[s], a, self.idx[self._mover(s, a_real)]] += pr
        return P
    def R(self): return np.array([self.recompensa(s) for s in self.estados])
    def es_terminal(self): return np.array([s in self.terminal for s in self.estados])

env = GridWorld()
def iteracion_de_valores(env, tol=1e-8):
    V = np.zeros(env.nS); R = env.R(); term = env.es_terminal(); hist = []
    while True:
        # Q(s,a) = sum_s' P(s'|s,a) [ r(s') + gamma V(s') ] ; en terminales el valor es solo su recompensa
        Q = env.P @ (R + env.gamma * V * ~term)
        V_new = np.where(term, R, Q.max(1)); hist.append(np.abs(V_new - V).max())
        if hist[-1] < tol: return V_new, Q.argmax(1), hist
        V = V_new
V_opt, pi_opt, hist = iteracion_de_valores(env)
print(f"convergió en {len(hist)} iteraciones")

def dibujar(env, V, pi, ax, titulo):
    grid = np.full((env.H, env.W), np.nan)
    for s, k in env.idx.items(): grid[s] = V[k]
    ax.imshow(grid, cmap="RdYlGn", vmin=-1, vmax=1)
    for s, k in env.idx.items():
        if s in env.terminal: ax.text(s[1], s[0], f"{V[k]:+.2f}", ha="center", va="center", fontweight="bold"); continue
        ax.text(s[1], s[0], "↑→↓←"[pi[k]] + f"\n{V[k]:.2f}", ha="center", va="center", fontsize=8)
    for m in env.muros: ax.add_patch(plt.Rectangle((m[1] - .5, m[0] - .5), 1, 1, color="k"))
    ax.set_title(titulo); ax.set_xticks([]); ax.set_yticks([])
fig, ax = plt.subplots(figsize=(6, 4.5)); dibujar(env, V_opt, pi_opt, ax, "v* y π* por iteración de valores (viento 20%)"); plt.show()

convergió en 46 iteraciones


**Observa** cómo la política óptima *rodea* el pozo por seguridad cuando hay ruido: el agente valora el riesgo a través de la esperanza. Cambia `ruido=0` y verás la ruta directa.

## 2. Aprender sin modelo: diferencias temporales (Sutton & Barto Cap. 6)

Normalmente **no conocemos $p$**. Los métodos TD aprenden de la experiencia $(s,a,r,s')$ actualizando hacia un objetivo *bootstrap*:

* **SARSA** (on-policy): $Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma Q(s',a')-Q(s,a)\big]$ con $a'$ la acción que *realmente* tomará.
* **Q-learning** (off-policy): $Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big]$ — aprende la política óptima aunque explore con ε-greedy.

La diferencia entre corchetes es el **error TD** $\delta$: la señal de aprendizaje (y, según la neurociencia, lo que codifica la dopamina). Usamos `FrozenLake` de Gymnasium (resbaladizo: transiciones estocásticas).

In [3]:
if GYM:
    lake = gym.make("FrozenLake-v1", is_slippery=True)
    def td_control(env, algoritmo="qlearning", episodios=6000, alpha=0.1, gamma=0.99, eps0=1.0, eps_min=0.05, seed=0):
        r = np.random.default_rng(seed); nS, nA = env.observation_space.n, env.action_space.n
        Q = np.zeros((nS, nA)); exitos = []
        eps_greedy = lambda s, eps: r.integers(nA) if r.random() < eps else int(np.argmax(Q[s] + 1e-9 * r.random(nA)))
        for ep in range(episodios):
            eps = max(eps_min, eps0 * (1 - ep / (0.7 * episodios)))
            s, _ = env.reset(seed=int(r.integers(1e9))); a = eps_greedy(s, eps); done = False; total = 0
            while not done:
                s2, rew, term, trunc, _ = env.step(a); done = term or trunc; total += rew
                a2 = eps_greedy(s2, eps)
                objetivo = rew + gamma * (0 if term else (Q[s2].max() if algoritmo == "qlearning" else Q[s2, a2]))
                Q[s, a] += alpha * (objetivo - Q[s, a])        # error TD
                s, a = s2, a2
            exitos.append(total)
        return Q, np.array(exitos)
    t0 = time.time(); Q_ql, ex_ql = td_control(lake, "qlearning"); Q_sa, ex_sa = td_control(lake, "sarsa"); print(f"{time.time()-t0:.1f}s")
    suav = lambda x, w=300: np.convolve(x, np.ones(w) / w, mode="valid")
    plt.plot(suav(ex_ql), label="Q-learning"); plt.plot(suav(ex_sa), label="SARSA"); plt.xlabel("episodio"); plt.ylabel("tasa de éxito (media móvil)")
    plt.title("FrozenLake resbaladizo (óptimo ≈ 0.74-0.82)"); plt.legend(); plt.show()

    # evaluación greedy sin exploración
    def evaluar(env, Q, n=2000):
        ok = 0
        for _ in range(n):
            s, _ = env.reset(); done = False
            while not done:
                s, rew, term, trunc, _ = env.step(int(Q[s].argmax())); done = term or trunc
            ok += rew
        return ok / n
    print(f"tasa de éxito greedy: Q-learning={evaluar(lake, Q_ql):.3f}  SARSA={evaluar(lake, Q_sa):.3f}")
    print("política Q-learning (4x4):"); print(np.array(list("←↓→↑"))[Q_ql.argmax(1)].reshape(4, 4))

4.3s


tasa de éxito greedy: Q-learning=0.749  SARSA=0.721
política Q-learning (4x4):
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]


## 3. Deep Q-Network (DQN) (Géron Cap. 18; Mnih et al., 2015)

Con estados continuos (`CartPole`: posición, velocidad, ángulo, velocidad angular) no cabe una tabla: aproximamos $Q_\theta(s,a)$ con una red. Dos trucos hacen que el aprendizaje TD con redes sea estable:
1. **Replay buffer**: almacenar transiciones y muestrear lotes aleatorios → rompe la correlación temporal y reutiliza datos.
2. **Red objetivo** $Q_{\theta^-}$ congelada (copiada cada $N$ pasos): el objetivo $r+\gamma\max_{a'}Q_{\theta^-}(s',a')$ deja de moverse con cada actualización.

Pérdida: Huber sobre el error TD. Mejoras habituales: **Double DQN** (desacoplar selección y evaluación del máximo → reduce sobreestimación), *dueling*, *prioritized replay*, *n-step*. Entrenamos en CPU en ~1-2 min (limitado a pocos episodios para la validación del notebook; sube `episodios` para llegar a 500 de recompensa).

In [4]:
if GYM and TORCH:
    env_cp = gym.make("CartPole-v1")
    class QNet(nn.Module):
        def __init__(self, nS, nA): super().__init__(); self.net = nn.Sequential(nn.Linear(nS, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, nA))
        def forward(self, x): return self.net(x)

    def dqn(env, episodios=150, gamma=0.99, lr=1e-3, batch=64, buffer=20000, target_every=500, eps_decay=4000, double=True, seed=0):
        torch.manual_seed(seed); r = np.random.default_rng(seed)
        nS, nA = env.observation_space.shape[0], env.action_space.n
        q, q_tgt = QNet(nS, nA), QNet(nS, nA); q_tgt.load_state_dict(q.state_dict()); opt = torch.optim.Adam(q.parameters(), lr)
        mem = deque(maxlen=buffer); pasos = 0; retornos = []
        for ep in range(episodios):
            s, _ = env.reset(seed=int(r.integers(1e9))); done = False; ret = 0
            while not done:
                eps = max(0.05, 1 - pasos / eps_decay)
                a = env.action_space.sample() if r.random() < eps else int(q(torch.tensor(s, dtype=torch.float32)).argmax())
                s2, rew, term, trunc, _ = env.step(a); done = term or trunc; ret += rew
                mem.append((s, a, rew, s2, float(term))); s = s2; pasos += 1
                if len(mem) >= 1000:
                    idx = r.choice(len(mem), batch, replace=False); S, A_, R_, S2, T = [torch.tensor(np.array(x), dtype=torch.float32) for x in zip(*[mem[i] for i in idx])]
                    with torch.no_grad():
                        if double: a_star = q(S2).argmax(1, keepdim=True); q_next = q_tgt(S2).gather(1, a_star).squeeze(1)
                        else: q_next = q_tgt(S2).max(1).values
                        objetivo = R_ + gamma * (1 - T) * q_next
                    q_sa = q(S).gather(1, A_.long()[:, None]).squeeze(1)
                    loss = F.smooth_l1_loss(q_sa, objetivo); opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q.parameters(), 10); opt.step()
                if pasos % target_every == 0: q_tgt.load_state_dict(q.state_dict())
            retornos.append(ret)
        return q, np.array(retornos)
    t0 = time.time(); q_net, ret_dqn = dqn(env_cp, episodios=150); print(f"DQN: {time.time()-t0:.0f}s, retorno medio últimos 20 episodios = {ret_dqn[-20:].mean():.0f} (máx 500)")
    plt.plot(ret_dqn, alpha=0.4); plt.plot(np.convolve(ret_dqn, np.ones(10) / 10, mode="valid"), lw=2); plt.xlabel("episodio"); plt.ylabel("retorno"); plt.title("Double DQN en CartPole-v1"); plt.show()

DQN: 326s, retorno medio últimos 20 episodios = 190 (máx 500)


## 4. Gradiente de política: REINFORCE y actor-crítico (Sutton & Barto Cap. 13)

En vez de aprender valores y derivar la política, parametrizamos $\pi_\theta(a|s)$ y ascendemos por el gradiente del retorno esperado. **Teorema del gradiente de política**:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\Big[\sum_t \nabla_\theta\log\pi_\theta(A_t|S_t)\,G_t\Big].$$
Intuición: aumenta la probabilidad de las acciones seguidas de retornos altos. **REINFORCE** usa el retorno Monte Carlo $G_t$ (alta varianza). Restar una **línea base** $b(s)$ no sesga y reduce la varianza; con $b=V(s)$ el factor $G_t - V(s_t)$ es la **ventaja** → **actor-crítico** (A2C). Ventajas de PG: acciones continuas, políticas estocásticas, convergencia suave.

**PPO** (Schulman et al., 2017), el caballo de batalla desde robótica hasta RLHF, limita el cambio de política por paso con un objetivo recortado:
$$L^{CLIP}=\mathbb{E}\big[\min\big(\rho_t\hat A_t,\ \mathrm{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t\big)\big],\quad \rho_t=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}.$$

In [5]:
if GYM and TORCH:
    def reinforce(env, episodios=300, gamma=0.99, lr=2e-3, baseline=True, seed=0):
        torch.manual_seed(seed); r = np.random.default_rng(seed)
        nS, nA = env.observation_space.shape[0], env.action_space.n
        actor = nn.Sequential(nn.Linear(nS, 64), nn.Tanh(), nn.Linear(64, nA)); critico = nn.Sequential(nn.Linear(nS, 64), nn.Tanh(), nn.Linear(64, 1))
        opt = torch.optim.Adam(list(actor.parameters()) + list(critico.parameters()), lr); retornos = []
        for ep in range(episodios):
            s, _ = env.reset(seed=int(r.integers(1e9))); done = False; logps, vals, rews = [], [], []
            while not done:
                st = torch.tensor(s, dtype=torch.float32); dist = torch.distributions.Categorical(logits=actor(st)); a = dist.sample()
                s, rew, term, trunc, _ = env.step(int(a)); done = term or trunc
                logps.append(dist.log_prob(a)); vals.append(critico(st).squeeze()); rews.append(rew)
            G = np.zeros(len(rews)); acc = 0
            for t in reversed(range(len(rews))): acc = rews[t] + gamma * acc; G[t] = acc
            G = torch.tensor(G, dtype=torch.float32); V = torch.stack(vals); logp = torch.stack(logps)
            ventaja = (G - V.detach()) if baseline else G
            ventaja = (ventaja - ventaja.mean()) / (ventaja.std() + 1e-8)           # normalización: reduce aún más la varianza
            loss = -(logp * ventaja).mean() + (0.5 * F.mse_loss(V, G) if baseline else 0)
            opt.zero_grad(); loss.backward(); opt.step(); retornos.append(sum(rews))
        return np.array(retornos)
    t0 = time.time(); ret_ac = reinforce(env_cp, baseline=True); ret_rf = reinforce(env_cp, baseline=False); print(f"{time.time()-t0:.0f}s")
    sm = lambda x: np.convolve(x, np.ones(15) / 15, mode="valid")
    plt.plot(sm(ret_rf), label="REINFORCE"); plt.plot(sm(ret_ac), label="REINFORCE + línea base (actor-crítico MC)"); plt.legend()
    plt.xlabel("episodio"); plt.ylabel("retorno (media móvil)"); plt.title("Gradiente de política en CartPole"); plt.show()

66s


## 5. RL para modelos de lenguaje: RLHF, DPO, GRPO (Fregly Cap. 7)

El MDP de un LLM: estado = *prompt* + tokens generados; acción = siguiente token; recompensa = al final de la respuesta. Tres generaciones de métodos:

1. **RLHF con PPO** (Ouyang et al., 2022): (a) recoger comparaciones humanas entre respuestas; (b) entrenar un **modelo de recompensa** $r_\phi$ (Bradley–Terry: $P(y_1\succ y_2)=\sigma(r(y_1)-r(y_2))$); (c) optimizar la política con PPO maximizando $r_\phi - \beta\,\mathrm{KL}(\pi_\theta\|\pi_{ref})$ (la KL evita que "haga trampa" al modelo de recompensa).
2. **DPO** (Rafailov et al., 2023): elimina el modelo de recompensa y PPO; optimiza directamente sobre pares preferidos/rechazados:
$$\mathcal L_{DPO} = -\mathbb{E}\Big[\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\Big)\Big].$$
Más simple y estable; hoy el estándar para alineación por preferencias (junto a variantes: IPO, KTO, ORPO, SimPO).
3. **GRPO / RLVR** (DeepSeekMath 2024, DeepSeek-R1 2025): para tareas con **recompensa verificable** (matemáticas, código, formato) se muestrean $G$ respuestas por prompt y la ventaja de cada una es su recompensa **normalizada dentro del grupo** — sin crítico. Es lo que produce los modelos "razonadores" con cadenas de pensamiento largas.

Implementamos DPO y GRPO **en miniatura** sobre una política tabular de un "bandit con contexto" para ver la mecánica de la pérdida.

In [6]:
if TORCH:
    # Política sobre 6 'respuestas' posibles a un prompt; la referencia es uniforme. Preferencias humanas simuladas: 4 > 5 > 3 > otras.
    calidad = torch.tensor([0.0, 0.2, 0.1, 1.0, 3.0, 2.0])
    ref_logits = torch.zeros(6)
    def dpo_paso(theta, pares, beta=0.5, lr=0.5):
        logp = theta.log_softmax(0); logp_ref = ref_logits.log_softmax(0)
        yw, yl = pares[:, 0], pares[:, 1]
        margen = beta * ((logp[yw] - logp_ref[yw]) - (logp[yl] - logp_ref[yl]))
        loss = -F.logsigmoid(margen).mean(); g, = torch.autograd.grad(loss, theta); return theta - lr * g, loss.item()
    # generar preferencias con ruido (Bradley-Terry sobre 'calidad')
    pares = []
    for _ in range(400):
        i, j = np.random.default_rng().choice(6, 2, replace=False); p = torch.sigmoid(calidad[i] - calidad[j]).item()
        pares.append((i, j) if np.random.rand() < p else (j, i))
    pares = torch.tensor(pares)
    theta = torch.zeros(6, requires_grad=True)
    for it in range(200): theta, l = dpo_paso(theta, pares); theta = theta.detach().requires_grad_(True)
    print("DPO  -> política:", theta.softmax(0).detach().numpy().round(3), " (mayor masa en 4, luego 5, 3)")

    # GRPO: muestrea G respuestas, recompensa verificable = 1 si la respuesta es la 'correcta' (4), ventaja normalizada por grupo
    theta_g = torch.zeros(6, requires_grad=True); opt = torch.optim.Adam([theta_g], 0.1)
    for it in range(150):
        dist = torch.distributions.Categorical(logits=theta_g); acciones = dist.sample((16,))       # grupo de G=16
        R = (acciones == 4).float(); A_ = (R - R.mean()) / (R.std() + 1e-6)                            # ventaja relativa al grupo (sin crítico)
        kl = (theta_g.softmax(0) * (theta_g.log_softmax(0) - ref_logits.log_softmax(0))).sum()          # penalización KL a la referencia
        loss = -(dist.log_prob(acciones) * A_).mean() + 0.02 * kl; opt.zero_grad(); loss.backward(); opt.step()
    print("GRPO -> política:", theta_g.softmax(0).detach().numpy().round(3), " (concentra en la respuesta verificable 4)")

DPO  -> política: [0.005 0.017 0.013 0.045 0.644 0.276]  (mayor masa en 4, luego 5, 3)
GRPO -> política: [0.    0.    0.    0.    0.998 0.   ]  (concentra en la respuesta verificable 4)


### Práctica real (2026)
```python
# OPCIONAL — uv pip install trl peft
from trl import DPOTrainer, DPOConfig, GRPOTrainer, GRPOConfig
# DPO: dataset con columnas prompt / chosen / rejected
# GRPO: reward_funcs=[lambda completions, **kw: [1.0 if verificar(c) else 0.0 for c in completions]]
```
* Simuladores: **MuJoCo**, **Isaac Lab** (robótica masivamente paralela en GPU), **PettingZoo** (multi-agente), **Minari** (RL *offline*).
* Librerías: **Stable-Baselines3** (fiable), **CleanRL** (implementaciones de una sola página, ideales para aprender), **RLlib** (escala), **TRL** (LLMs).
* Fronteras: RL *offline* (CQL, IQL, Decision Transformer), modelos del mundo (Dreamer v3), RL para agentes de software y uso de herramientas (recompensas por tarea completada).

## 📦 Recursos de los repositorios oficiales de los libros

| Libro | Recurso | Para qué usarlo aquí |
|---|---|---|
| Sutton & Barto | Libro gratuito: <http://incompleteideas.net/book/the-book-2nd.html> · código oficial por capítulo (Lisp/Python): <http://incompleteideas.net/book/code/code2nd.html> · **réplica Python de casi todas las figuras**: [ShangtongZhang/reinforcement-learning-an-introduction](https://github.com/ShangtongZhang/reinforcement-learning-an-introduction/tree/master/) — [`chapter03/`](https://github.com/ShangtongZhang/reinforcement-learning-an-introduction/tree/master/chapter03) gridworld, [`chapter04/`](https://github.com/ShangtongZhang/reinforcement-learning-an-introduction/tree/master/chapter04) (Jack's car rental, gambler), [`chapter05/`](https://github.com/ShangtongZhang/reinforcement-learning-an-introduction/tree/master/chapter05) blackjack, [`chapter06/`](https://github.com/ShangtongZhang/reinforcement-learning-an-introduction/tree/master/chapter06) (random walk, **windy gridworld**, **cliff walking**, Q-learning vs Double Q), [`chapter13/`](https://github.com/ShangtongZhang/reinforcement-learning-an-introduction/tree/master/chapter13) (REINFORCE, short corridor) | `chapter06/cliff_walking.py` es la solución del ejercicio 3; `chapter13/short_corridor.py` reproduce la figura 13.1 y muestra por qué a veces se necesita una política estocástica |
| Géron, *Hands-On ML 3* | [`18_reinforcement_learning.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/18_reinforcement_learning.ipynb) | Políticas neuronales, REINFORCE, MDP, TD, **DQN con Keras y TF-Agents**; usa `gym` → sustituye por `gymnasium` (API `terminated/truncated`) |
| Zhang et al., *D2L* | [Cap. 17 Reinforcement Learning](https://d2l.ai/chapter_reinforcement-learning/index.html) (MDP, iteración de valores, Q-learning con Gymnasium) | Breve y con código PyTorch |
| Fregly et al. | [`07_rlhf/`](https://github.com/generative-ai-on-aws/generative-ai-on-aws/tree/main/07_rlhf) (modelo de recompensa + PPO con `trl` sobre Llama 2 en SageMaker) | La implementación real de la sección 5 |
| Otros | [CleanRL](https://docs.cleanrl.dev) (DQN, PPO, SAC en ficheros únicos), [Spinning Up](https://spinningup.openai.com), [Gymnasium docs](https://gymnasium.farama.org) | Implementaciones de referencia legibles para el ejercicio 5 (PPO) |

## 6. Ejercicios
1. **Iteración de políticas.** Implementa evaluación de políticas + mejora *greedy* (Sutton & Barto 4.3) sobre `GridWorld` y compara el número de barridos con iteración de valores.
2. **γ y riesgo.** Resuelve el gridworld con `gamma` ∈ {0.5, 0.9, 0.99} y `ruido` ∈ {0, 0.2, 0.4}. ¿Cuándo la política prefiere el camino largo y seguro?
3. **SARSA vs Q-learning en el acantilado.** Implementa `CliffWalking-v1` de Gymnasium: Q-learning aprende la ruta óptima al borde del acantilado pero *durante el aprendizaje* cae más (Sutton & Barto ej. 6.6). Reprodúcelo.
4. **DQN completo.** Aumenta `episodios` a 400, añade *dueling network* y compara Double DQN vs DQN vanilla en 3 semillas (media ± desviación).
5. **PPO.** Extiende el actor-crítico a PPO: recoge 2048 pasos, calcula ventajas con GAE ($\lambda=0.95$), haz 4 épocas de minibatches con el objetivo recortado ($\epsilon=0.2$). Compara estabilidad con REINFORCE.
6. **Reward hacking.** En la simulación DPO/GRPO, introduce una respuesta (índice 1) que el "modelo de recompensa" puntúa alto por error. ¿Qué pasa con y sin el término KL? Discute la analogía con LLMs que aprenden a adular al juez.

## Referencias
* Sutton, R. S. & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2.ª ed.). MIT Press. Gratuito: http://incompleteideas.net/book/the-book-2nd.html. Cap. 3-6, 13.
* Géron, A. (2022). *Hands-On ML* (3.ª ed.). Cap. 18.
* Zhang, A. et al. *Dive into Deep Learning*. Cap. 17.
* Fregly, C. et al. (2023). *Generative AI on AWS*. Cap. 7.
* Mnih, V. et al. (2015). *Human-level control through deep reinforcement learning* (DQN). Nature. · van Hasselt, H. et al. (2016). *Double DQN*. AAAI.
* Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms*. · Schulman, J. et al. (2016). *GAE*.
* Ouyang, L. et al. (2022). *InstructGPT*. · Rafailov, R. et al. (2023). *DPO*. NeurIPS. · Shao, Z. et al. (2024). *DeepSeekMath* (GRPO). · DeepSeek-AI (2025). *DeepSeek-R1*. Nature.
* Gymnasium: https://gymnasium.farama.org · CleanRL: https://docs.cleanrl.dev · Spinning Up in Deep RL (OpenAI).